## CITS2402 - Introduction to Data Science - Project
### Semester 2, 2026

### Household Composition and Rental Housing Affordability: Australia (2021) vs New Zealand (2023)

**Date:** [Insert submission date]


## Declaration

This declaration should be completed and remain attached to the top of your submission.

**We are aware of the University's [policy on academic conduct](https://www.uwa.edu.au/policy/-/media/project/uwa/uwa/policy-library/policy/student-administration/academic-integrity/academic-integrity-policy.doc) and declare that this project is entirely the work of the authors listed below and that suitable acknowledgement has been made for any sources of information used in preparing it. We have retained a copy for our own records.**

- Name 1: [Insert name]
- Student ID 1: [Insert student number]
- Name 2: [Insert name]
- Student ID 2: [Insert student number]
- Name 3: [Insert name, delete if a team of 2]
- Student ID 3: [Insert student number, delete if a team of 2]
- Date: [Insert date]


Rename this file to `CITS2402-Project-STDNO1-STDNO2.ipynb` (or `-STDNO1-STDNO2-STDNO3.ipynb` for a team of three) before submitting, replacing the suffix with your actual student numbers.

<hr>


## **I. Introduction**

**1.1 Aim**

This report aims to examine the relationship between household/family composition and housing affordability in Australia and New Zealand, using weekly rent as a key affordability indicator. As housing costs continue to rise in both countries, understanding which household types bear the greatest rental burden—and whether this burden is distributed similarly across both nations—offers insight into broader social and economic pressures shaping each country's housing landscape. By comparing rent patterns across household types in each country's most recent census, this report seeks to identify not only how much rent varies by household type, but where these pressures are most acute, and whether household composition itself is linked to housing cost burden.

**1.2 Research Questions**

This report will answer the following questions:
1. Research Question 1: *How does median weekly rent differ across household/family types (e.g., couple-only, couple-with-children, one-parent, lone-person, group households) in Australia (2021) and New Zealand (2023), and which household types face the highest and lowest rent burdens in each country?*
2. Research Question 2: *How does this rent-by-household-type pattern vary regionally (e.g., major cities vs. regional/rural areas) within each country, and which household types show the greatest rent disparity between urban and regional areas?*
3. Research Question 3: *Do household types with higher shares in urban areas (from RQ2) also face disproportionately higher rent burdens — and is this relationship consistent between Australia and New Zealand, or does it diverge?*


**1.3 Hypotheses**

State one falsifiable prediction per research question, before you look at the results. Keep each to 1-2 sentences and be specific enough that a reader could tell, from your later results, whether it was supported or rejected (see the worked example in the marking guide for the tone to aim for).

1. **Research Question 1:**
    * **Hypothesis:** [TODO — e.g. which household type(s) you expect to carry the heaviest rent burden in each country, and why]

2. **Research Question 2:**
    * **Hypothesis:** [TODO — e.g. which household type(s) you expect to show the biggest urban/regional rent gap, and why]

3. **Research Question 3:**
    * **Hypothesis:** [TODO — e.g. whether you expect the urban-share → rent-burden link to hold in both countries or to diverge]


## **II. Data Preparation and Exploratory Data Analysis (EDA)**

**2.1 Context and Aim**

[TODO — 1-2 sentences: this section shows how the ABS and Stats NZ census data on rent and household composition were sourced and prepared for analysis.]

**2.2 Dataset Overview**

1. **Source (AUS)**: Australian Bureau of Statistics (ABS), 2021 Census — [TODO: TableBuilder Pro export, once confirmed]
2. **Source (NZ)**: Stats NZ, 2023 Census — [TODO: Aotearoa Data Explorer export, once confirmed]
3. **Years**: 2021 (Australia), 2023 (New Zealand)
4. **Population**: [TODO — e.g. occupied private dwellings that are rented]
5. **Unit**: [TODO — e.g. weekly rent in local currency, count of households]
6. **Variables/tables used**:
     * AUS: [TODO — exact TableBuilder variable codes, e.g. household/family composition × rent (weekly)]
     * NZ: [TODO — exact Aotearoa Data Explorer variable names]


**2.3 Procedure**

*Australia (ABS TableBuilder):*
1. Register at the [ABS Registration Centre](https://www.abs.gov.au/statistics/microdata-tablebuilder/tablebuilder) using your university email address.
2. Confirm whether you were granted Census TableBuilder **Pro** access (check 'My profile summary'); request it from microdata.access@abs.gov.au if not.
3. Log in to TableBuilder and select the 2021 Census dataset that counts households by place of usual residence.
4. Add [household/family composition] to rows and [rent (weekly)] to columns; set geography to Australia.
5. Download the table as CSV.
6. [TODO: add the regional breakdown steps you use for RQ2 here.]

*New Zealand (Stats NZ Aotearoa Data Explorer):*
1. [TODO — fill in the equivalent click-path once confirmed: navigate to the Aotearoa Data Explorer, select the relevant census dataset, choose household composition and weekly rent as variables, export CSV.]

Place all exported CSVs in the same folder as this notebook before running the cleaning code below.

**2.4 Assumptions for Data Preparation**

1. [TODO] Currency: NZD figures are converted to AUD using [exchange rate / PPP — state which, the source, and the reference date].
2. [TODO] 'Not stated' and 'not applicable' rent/household categories are excluded from percentage calculations.
3. [TODO] Australian and New Zealand household/family composition categories are harmonised into a common set of labels (see `HOUSEHOLD_TYPE_MAP_AU` / `HOUSEHOLD_TYPE_MAP_NZ` below); note any categories that don't map cleanly.
4. TableBuilder and Aotearoa Data Explorer both apply small random perturbation to cell counts for confidentiality, which may create minor discrepancies in totals — acceptable within a stated tolerance.
5. [TODO] Any other assumptions specific to your topic.


**2.5 Exploratory Data Analysis Checks (planned)**

1. Both tables contain the expected columns with correct data types and no unexpected fields.
2. Household-type labels match exactly (after harmonisation) across the two countries.
3. Counts/percentages sum to approximately 100% within each country, within a stated tolerance.


## **III. Data Cleaning**

**3.1 Context and Aim**

[TODO — 1-2 sentences describing what the cleaning functions below do and why.]

**3.2 Code Setup**

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np

# ---------------------------------------------------------------------------
# File paths (RELATIVE paths only).
# All data files must be submitted in the same directory as this notebook.
# Do NOT mount Google Drive (drive.mount(...)) -- a marker running this in a
# fresh Colab kernel will not have access to your personal Drive folder.
# ---------------------------------------------------------------------------
AU_RENT_FILE = "au_rent_by_household_2021.csv"   # TODO: rename to match your actual TableBuilder export
NZ_RENT_FILE = "nz_rent_by_household_2023.csv"   # TODO: rename to match your actual Stats NZ export

AU_REGIONAL_FILE = "au_rent_by_household_region_2021.csv"  # TODO: for RQ2, if a separate file
NZ_REGIONAL_FILE = "nz_rent_by_household_region_2023.csv"  # TODO: for RQ2, if a separate file

# ---------------------------------------------------------------------------
# Household type harmonisation.
# TODO: once you have the real category labels from TableBuilder and the
# Aotearoa Data Explorer, fill these in so both countries share one common
# set of household-type labels for comparison.
# ---------------------------------------------------------------------------
HOUSEHOLD_TYPE_MAP_AU = {
    # "ABS category label": "Common label",
    # e.g. "Couple family with no children": "Couple only",
}

HOUSEHOLD_TYPE_MAP_NZ = {
    # "Stats NZ category label": "Common label",
    # e.g. "Couple only with two usual residents": "Couple only",
}

COMMON_HOUSEHOLD_TYPES = [
    "Couple only",
    "Couple with children",
    "One parent with children",
    "Lone person",
    "Group household",
    # TODO: adjust once you've confirmed exact categories in both datasets
]

# ---------------------------------------------------------------------------
# Currency conversion.
# TODO: decide + justify your approach in 2.4 Assumptions (exchange rate vs
# PPP), then set the constant below and cite your source.
# ---------------------------------------------------------------------------
AUD_PER_NZD = None  # TODO: set this once you've sourced a conversion rate

def convert_nzd_to_aud(amount_nzd, rate=AUD_PER_NZD):
    """Convert an NZD amount to AUD using a fixed exchange/PPP rate."""
    if rate is None:
        raise ValueError("Set AUD_PER_NZD before converting currency.")
    return amount_nzd * rate


**3.3 Cleaning Raw Data**

**3.3.1 Cleaning Australian Data (Rent by Household/Family Composition)**

In [ ]:
def clean_au_data(file_path, household_map):
    """
    Load and tidy the Australian TableBuilder export of Rent (weekly) by
    Household/Family Composition.

    TODO:
      - Read the raw CSV.
      - Rename ABS category labels to the common household-type labels
        using `household_map`.
      - Convert rent to a numeric column (AUD).
      - Return a tidy dataframe with columns:
        ['household_type', 'rent_aud', 'count'] (or ['household_type', 'median_rent_aud']
        if TableBuilder computed the median for you directly).
    """
    data = pd.read_csv(file_path)
    # TODO: implement cleaning steps here
    raise NotImplementedError("Fill in the AU cleaning steps once you have the real TableBuilder export.")

# au_rent = clean_au_data(AU_RENT_FILE, HOUSEHOLD_TYPE_MAP_AU)
# print(au_rent)


**3.3.2 Cleaning New Zealand Data (Rent by Household Composition)**

In [ ]:
def clean_nz_data(file_path, household_map):
    """
    Load and tidy the Stats NZ Aotearoa Data Explorer export of Weekly Rent
    by Household Composition.

    TODO:
      - Read the raw CSV.
      - Rename Stats NZ category labels to the common household-type labels
        using `household_map`.
      - Keep rent in NZD for now (convert later, once combining datasets).
      - Return a tidy dataframe with columns:
        ['household_type', 'rent_nzd', 'count'] (or ['household_type', 'median_rent_nzd']).
    """
    data = pd.read_csv(file_path)
    # TODO: implement cleaning steps here
    raise NotImplementedError("Fill in the NZ cleaning steps once you have the real Stats NZ export.")

# nz_rent = clean_nz_data(NZ_RENT_FILE, HOUSEHOLD_TYPE_MAP_NZ)
# print(nz_rent)


**3.4 EDA Checks**

**3.4.1 Context and Data Types Check**

In [ ]:
# TODO: once au_rent / nz_rent exist, check expected columns and dtypes.
# EXPECTED_COLUMNS = ['household_type', 'rent_aud', 'count']  # adjust to your actual schema
#
# for name, df in [("au_rent", au_rent), ("nz_rent", nz_rent)]:
#     missing = [c for c in EXPECTED_COLUMNS if c not in df.columns]
#     assert not missing, f"{name}: missing columns {missing}"
#     assert pd.api.types.is_numeric_dtype(df['count']), f"{name}: 'count' not numeric"
#
# print("Both tables have the expected columns and correct data types.")


**3.4.2 Row and Label Check**

In [ ]:
# TODO: confirm both countries share the same set of harmonised household-type
# labels after mapping, and flag any category that didn't map cleanly.
#
# assert set(au_rent['household_type']) == set(COMMON_HOUSEHOLD_TYPES), "AU: unexpected household types"
# assert set(nz_rent['household_type']) == set(COMMON_HOUSEHOLD_TYPES), "NZ: unexpected household types"
#
# print("Household-type labels match the common set in both countries.")


**3.4.3 Totals and Sums Check**

In [ ]:
# TODO: set a tolerance and confirm household-type counts sum to (approximately)
# the reported national total in each country, following the same pattern as
# the totals-vs-age-bands check used for the occupation-shares example.
#
# PCT_TOL = 0.15  # percentage points
#
# for name, df in [("au_rent", au_rent), ("nz_rent", nz_rent)]:
#     share_sum = (df['count'] / df['count'].sum() * 100).sum()
#     assert abs(share_sum - 100) <= PCT_TOL, f"{name}: shares sum to {share_sum:.2f}%"
#
# print("Household-type shares sum to ~100% in both countries, within tolerance.")


## **IV. Data Analysis**

**4.1 Context and Aim**

[TODO — 1-2 sentences on how the cleaned data will be combined and visualised to answer each RQ.]

**4.2 Data Interpretation**

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# TODO: build one tidy, combined dataframe (both countries, common household
# types, rent all in AUD) that every RQ below can filter/reshape from.
#
# combined = pd.concat([
#     au_rent.assign(country='Australia'),
#     nz_rent.assign(country='New Zealand', rent_aud=convert_nzd_to_aud(nz_rent['rent_nzd'])),
# ], ignore_index=True)
# print(combined.head())


**4.2.1 Research Question 1**:

*How does median weekly rent differ across household/family types (e.g., couple-only, couple-with-children, one-parent, lone-person, group households) in Australia (2021) and New Zealand (2023), and which household types face the highest and lowest rent burdens in each country?*

In [ ]:
# TODO: grouped bar chart -- household_type on the x-axis, median weekly rent
# (AUD) on the y-axis, one bar per country. Sort households by (e.g.) combined
# rent so the chart reads highest-to-lowest burden. Add title, axis labels,
# legend, and value labels on each bar.

fig, ax = plt.subplots(figsize=(8, 5))
# ax.bar(...)
# ax.set_title(...)
# ax.set_xlabel(...); ax.set_ylabel(...)
# TODO: implement
plt.show()


**Analysis:**

[TODO — state the numeric findings, name the highest/lowest-burden household types in each country, answer RQ1 directly, and say whether your RQ1 hypothesis is supported or rejected.]

**4.2.2 Research Question 2**:

*How does this rent-by-household-type pattern vary regionally (e.g., major cities vs. regional/rural areas) within each country, and which household types show the greatest rent disparity between urban and regional areas?*

In [ ]:
# TODO: repeat the RQ1-style chart but split by region (major cities vs
# regional/rural) within each country, or plot the urban-minus-regional rent
# gap per household type directly (one bar per household type, coloured by
# country).

fig, ax = plt.subplots(figsize=(8, 5))
# TODO: implement
plt.show()


**Analysis:**

[TODO — name the household type(s) with the biggest urban/regional rent gap in each country, answer RQ2 directly, and say whether your RQ2 hypothesis is supported or rejected.]

**4.2.3 Research Question 3**:

*Do household types with higher shares in urban areas (from RQ2) also face disproportionately higher rent burdens — and is this relationship consistent between Australia and New Zealand, or does it diverge?*

In [ ]:
# TODO: scatter plot combining your RQ1 and RQ2 results -- e.g. x = share of
# a household type living in urban areas, y = rent burden (or urban-regional
# rent gap) for that household type, one point per household type, coloured
# by country, to test whether higher urban share lines up with higher burden.

fig, ax = plt.subplots(figsize=(8, 5))
# ax.scatter(...)
# TODO: implement
plt.show()


**Analysis:**

[TODO — state whether urban share and rent burden are linked, whether the pattern is consistent or diverges between Australia and New Zealand, answer RQ3 directly, and say whether your RQ3 hypothesis is supported or rejected.]

## **V. Summary**

[TODO — 3-5 sentences: which household types face the greatest rent burden in each country, how this shifts between urban and regional areas, and whether urban concentration and rent burden are linked — tying back to the Aim.]

**5.1 Recommendations**

[TODO — 1-2 practical implications of your findings, e.g. for housing policy or renters, in each country.]

**5.2 Limitations**

1. [TODO] Currency conversion (exchange rate vs PPP) affects the size, though not necessarily the direction, of any AUD/NZD rent gap.
2. [TODO] TableBuilder/Aotearoa Data Explorer confidentiality perturbation may introduce small inaccuracies in low-count household types.
3. [TODO] Household-type category harmonisation between the two countries is an approximation; note any categories that didn't map cleanly.
4. [TODO] Single census snapshot in each country (2021 vs 2023) — not a like-for-like reference date.
5. [TODO] Any other limitations specific to your analysis.
